In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_absolute_error
import xgboost as xgb

print("Loading 'pv_shading_dataset.csv'...")
df = pd.read_csv("pv_shading_dataset.csv")

print("Engineering physical features from raw electrical columns...")
epsilon = 1e-5

# feature engineering
df['Est_Current'] = df['Output_Power'] / (df['Output_Voltage'] + epsilon)
df['Static_Resistance'] = (df['Output_Voltage']**2) / (df['Output_Power'] + epsilon)
df['Power_V2_Ratio'] = df['Output_Power'] / (df['Output_Voltage']**2 + epsilon)
df['V_Midpoint_Deviation'] = df['Output_Voltage'] - 60.0
df['Current_Deficit'] = np.maximum(0.0, 8.0 - df['Est_Current'])
df['V_P_Interaction'] = df['Output_Voltage'] * df['Output_Power']
df['Normalized_Power'] = df['Output_Power'] / 270.0

feature_columns = [
    'Output_Voltage', 'Output_Power', 
    'Est_Current', 'Static_Resistance', 'Power_V2_Ratio', 
    'V_Midpoint_Deviation', 'Current_Deficit', 'V_P_Interaction', 'Normalized_Power'
]

X = df[feature_columns].values
y_shading = df[[f'Mod_{i}_G' for i in range(9)]].values   

y_config_raw = df['Optimal_Config_Label'].values                
label_encoder = LabelEncoder()
y_config = label_encoder.fit_transform(y_config_raw)

# Split dataset into training (80%) and validation/test (20%)
X_train, X_test, y_sh_train, y_sh_test, y_cfg_train, y_cfg_test = train_test_split(
    X, y_shading, y_config, test_size=0.2, random_state=42
)

print("\nTraining Model A: XGBoost Switch Configuration Classifier (100 Epochs Max)...")
config_model = xgb.XGBClassifier(
    n_estimators=100,            # Max rounds/epochs
    max_depth=10,
    learning_rate=0.08,
    objective='multi:softprob',
    early_stopping_rounds=10,    # Stop if validation loss doesn't improve for 10 epochs
    random_state=42,
    n_jobs=-1
)

# Fit classifier with early stopping evaluation set
config_model.fit(
    X_train, y_cfg_train,
    eval_set=[(X_test, y_cfg_test)],
    verbose=True  
)

print("\nTraining Model B: XGBoost 3x3 Module Shading Regressor...")
base_regressor = xgb.XGBRegressor(
    n_estimators=100,           
    max_depth=10,
    learning_rate=0.08,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)
shading_model = MultiOutputRegressor(base_regressor)

shading_model.fit(X_train, y_sh_train)

with open('config_xgb_model_eng.pkl', 'wb') as f:
    pickle.dump(config_model, f)
with open('shading_xgb_model_eng.pkl', 'wb') as f:
    pickle.dump(shading_model, f)
with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("\nModel training complete! Both models trained with an early-stopping monitor.")

cfg_predictions = config_model.predict(X_test)
cfg_accuracy = accuracy_score(y_cfg_test, cfg_predictions)

sh_predictions = shading_model.predict(X_test)
mae = mean_absolute_error(y_sh_test, sh_predictions)
shading_within_tolerance = np.mean(np.abs(y_sh_test - sh_predictions) <= 100.0)

print("\n======================= ENGINEERED XGBOOST REPORT =======================")
print(f"Optimal Epochs Utilized (Classifier)    : {config_model.best_iteration}")
print(f"Switch Configuration Prediction Accuracy : {cfg_accuracy * 100:.2f}%")
print(f"Module Insolation Prediction Accuracy    : {shading_within_tolerance * 100:.2f}% (within ±100 W/m²)")
print(f"Average Insolation Prediction Error       : {mae:.1f} W/m²")
print("=========================================================================")

Loading 'pv_shading_dataset.csv'...
Engineering physical features from raw electrical columns...

Training Model A: XGBoost Switch Configuration Classifier (100 Epochs Max)...
[0]	validation_0-mlogloss:1.20083
[1]	validation_0-mlogloss:1.11728
[2]	validation_0-mlogloss:1.04827
[3]	validation_0-mlogloss:0.98924
[4]	validation_0-mlogloss:0.93727
[5]	validation_0-mlogloss:0.89188
[6]	validation_0-mlogloss:0.85215
[7]	validation_0-mlogloss:0.81662
[8]	validation_0-mlogloss:0.78595
[9]	validation_0-mlogloss:0.75721
[10]	validation_0-mlogloss:0.73300
[11]	validation_0-mlogloss:0.71068
[12]	validation_0-mlogloss:0.69105
[13]	validation_0-mlogloss:0.67340
[14]	validation_0-mlogloss:0.65719
[15]	validation_0-mlogloss:0.64293
[16]	validation_0-mlogloss:0.63113
[17]	validation_0-mlogloss:0.61934
[18]	validation_0-mlogloss:0.61004
[19]	validation_0-mlogloss:0.60084
[20]	validation_0-mlogloss:0.59336
[21]	validation_0-mlogloss:0.58585
[22]	validation_0-mlogloss:0.57998
[23]	validation_0-mlogloss:0.